In [2]:
"""
03_model_calibration.py

Evaluates the clinical reliability of the 26-gene XGBoost model.
Calculates the Brier Score and generates a publication-ready Calibration Curve
(Reliability Diagram) on the independent holdout Vault (GSE65682).
"""

import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import xgboost as xgb
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

warnings.filterwarnings("ignore")

# ==========================================
# CONFIGURATION
# ==========================================
# Adjust BASE_DIR if running from inside the notebooks/ folder
BASE_DIR = Path.cwd().parent if "notebooks" in str(Path.cwd()) else Path(__file__).resolve().parents[2]
DATA_DIR = BASE_DIR / "data" / "processed" / "ml_tensors"
DEG_DIR = BASE_DIR / "data" / "processed" / "deg_tensors"
PLOT_DATA_DIR = BASE_DIR / "outputs" / "plot_data"
MODEL_DIR = BASE_DIR / "outputs" / "models"
FIG_OUT = BASE_DIR / "outputs" / "figures"

HOLDOUT_COHORT = 'GSE65682'

def main():
    print("[*] INITIATING CLINICAL RELIABILITY & CALIBRATION ANALYSIS...")
    FIG_OUT.mkdir(parents=True, exist_ok=True)

    # 1. Load the 26 Optimal Genes
    features_path = PLOT_DATA_DIR / "03_optimal_feature_list.csv"
    if not features_path.exists():
        raise FileNotFoundError("Optimal features list not found!")
    optimal_genes = pd.read_csv(features_path)['Optimal_Genes'].tolist()

    # 2. Load Tensors
    print("    -> Loading Master Tensors...")
    X_elite = pd.read_csv(DEG_DIR / "X_deg_master.csv.gz", compression='gzip')
    y = pd.read_csv(DATA_DIR / "y_master.csv")
    meta = pd.read_csv(DATA_DIR / "meta_master.csv")
    
    # 3. Isolate the Blind Vault Cohort
    test_mask = meta['Dataset'] == HOLDOUT_COHORT
    X_vault = X_elite.loc[test_mask, optimal_genes]
    
    target_col = 'Mortality' if 'Mortality' in y.columns else y.columns[0]
    y_vault = y.loc[test_mask, target_col].astype(int)

    # 4. Load the Locked Clinical Model
    model_path = MODEL_DIR / "sepsis_xgboost_26genes_clinical_v1.json"
    if not model_path.exists():
        raise FileNotFoundError(f"Model weights not found at {model_path}!")
        
    print(f"    -> Loading locked XGBoost weights for the 26-gene signature...")
    model = xgb.XGBClassifier()
    model.load_model(model_path)

    # 5. Predict Probabilities
    print("    -> Calculating predictive probabilities on the Holdout Vault...")
    vault_probs = model.predict_proba(X_vault)[:, 1]

    # 6. Calculate Calibration Metrics
    brier = brier_score_loss(y_vault, vault_probs)
    print(f"\n[*] CLINICAL RELIABILITY METRICS:")
    print(f"    -> Brier Score: {brier:.4f} (Ideal is 0.0, Baseline is ~0.25)")
    
    # Generate coordinates for the calibration curve (10 bins)
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_vault, vault_probs, n_bins=10, strategy='uniform'
    )

    # 7. Generate Publication Aesthetic Calibration Plot
    print("    -> Generating Reliability Diagram...")
    
    fig = plt.figure(figsize=(9, 9))
    gs = gridspec.GridSpec(4, 1, hspace=0.4)
    
    # TOP PANEL: Calibration Curve
    ax1 = plt.subplot(gs[:3, 0])
    
    # Ideal perfectly calibrated line
    ax1.plot([0, 1], [0, 1], linestyle='--', color='#888888', linewidth=1.5, label='Perfectly Calibrated')
    
    # Our Model's curve
    ax1.plot(mean_predicted_value, fraction_of_positives, marker='o', markersize=8,
             color='#db4325', linewidth=2.5, label=f'26-Gene XGBoost (Brier = {brier:.3f})')
    
    # Formatting Top Panel
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['left'].set_color('#aaaaaa')
    ax1.spines['bottom'].set_color('#aaaaaa')
    ax1.tick_params(colors='#555555')
    
    ax1.set_ylabel('True Probability in Patients (Fraction of Positives)', fontsize=12, color='#444444', labelpad=10)
    ax1.set_title('Model Calibration and Reliability (Vault Cohort: GSE65682)', fontsize=14, color='#333333', pad=15)
    ax1.legend(loc='upper left', frameon=False, fontsize=11)
    ax1.set_xlim([-0.05, 1.05])
    ax1.set_ylim([-0.05, 1.05])
    ax1.grid(True, linestyle=':', color='#f0f0f0')

    # BOTTOM PANEL: Histogram of predicted probabilities
    ax2 = plt.subplot(gs[3, 0], sharex=ax1)
    
    ax2.hist(vault_probs, bins=20, range=(0, 1), color='#4a6fe3', alpha=0.7, edgecolor='white', linewidth=1.2)
    
    # Formatting Bottom Panel
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.spines['left'].set_color('#aaaaaa')
    ax2.spines['bottom'].set_color('#aaaaaa')
    ax2.tick_params(colors='#555555')
    
    ax2.set_xlabel('Predicted Probability of Mortality', fontsize=12, color='#444444', labelpad=10)
    ax2.set_ylabel('Patient Count', fontsize=11, color='#444444')
    ax2.grid(axis='y', linestyle=':', color='#f0f0f0')

    plt.tight_layout()
    
    pdf_out = FIG_OUT / "Fig9_Calibration_Curve.pdf"
    plt.savefig(pdf_out, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"[*] SUCCESS! Calibration plot saved to {pdf_out.name}")

if __name__ == "__main__":
    main()

[*] INITIATING CLINICAL RELIABILITY & CALIBRATION ANALYSIS...
    -> Loading Master Tensors...
    -> Loading locked XGBoost weights for the 26-gene signature...
    -> Calculating predictive probabilities on the Holdout Vault...

[*] CLINICAL RELIABILITY METRICS:
    -> Brier Score: 0.2074 (Ideal is 0.0, Baseline is ~0.25)
    -> Generating Reliability Diagram...
[*] SUCCESS! Calibration plot saved to Fig9_Calibration_Curve.pdf
